In [6]:

import pandas as pd

cols = [
    "PlanId", "StateCode", "MetalLevel", "PlanType", 
    "monthly_premium", "drug_tier", "raw_copay", 
    "raw_coinsurance", "access_tier"
]

df = pd.read_csv("sql_export.csv", names=cols)

# Read the CSV exported from SSMS

In [7]:
print(df.head())

              PlanId StateCode MetalLevel PlanType  monthly_premium  \
0  10091OR0360004-00        OR     Bronze      PPO           315.86   
1  10091OR0360004-00        OR     Bronze      PPO           315.86   
2  10091OR0360004-00        OR     Bronze      PPO           315.86   
3  10091OR0360004-00        OR     Bronze      PPO           315.86   
4  10091OR0360004-00        OR     Bronze      PPO           315.86   

               drug_tier                   raw_copay raw_coinsurance  \
0          Generic Drugs                          $0              0%   
1          Generic Drugs  $20 Copay after deductible  Not Applicable   
2  Preferred Brand Drugs                          $0              0%   
3  Preferred Brand Drugs  $80 Copay after deductible  Not Applicable   
4        Specialty Drugs                          $0              0%   

                    access_tier  
0  Favorable Access (Low Copay)  
1              Moderate Barrier  
2  Favorable Access (Low Copay)  
3   

# Remove blank rows and very high outlier values


In [8]:
df = df.dropna(subset=["monthly_premium"])
df = df[df["monthly_premium"] > 0]
df = df[df["monthly_premium"] < 3000]

# Add monthly drug price based on drug tier


In [9]:
def get_drug_price(tier):
    if tier == "Generic Drugs":
        return 30
    elif tier == "Preferred Brand Drugs":
        return 350
    elif tier == "Specialty Drugs":
        return 4000
    else:
        return 0

df["benchmark_drug_cost"] = df["drug_tier"].apply(get_drug_price)

# Calculate how much the patient pays each month

In [10]:
patient_costs = []

for index, row in df.iterrows():
    if row["access_tier"] == "High Barrier (Coinsurance)":
        cost = row["benchmark_drug_cost"] * 0.30
    elif row["access_tier"] == "Moderate Barrier":
        cost = 45
    else:
        cost = 10
    patient_costs.append(cost)

df["est_patient_cost"] = patient_costs

# Check the output

In [11]:
print(df[["StateCode", "drug_tier", "access_tier", "est_patient_cost"]].head(10))

  StateCode              drug_tier                   access_tier  \
0        OR          Generic Drugs  Favorable Access (Low Copay)   
1        OR          Generic Drugs              Moderate Barrier   
2        OR  Preferred Brand Drugs  Favorable Access (Low Copay)   
3        OR  Preferred Brand Drugs              Moderate Barrier   
4        OR        Specialty Drugs  Favorable Access (Low Copay)   
5        OR        Specialty Drugs    High Barrier (Coinsurance)   
6        OR          Generic Drugs  Favorable Access (Low Copay)   
7        OR          Generic Drugs              Moderate Barrier   
8        OR  Preferred Brand Drugs  Favorable Access (Low Copay)   
9        OR  Preferred Brand Drugs              Moderate Barrier   

   est_patient_cost  
0              10.0  
1              45.0  
2              10.0  
3              45.0  
4              10.0  
5            1200.0  
6              10.0  
7              45.0  
8              10.0  
9              45.0  


# Save final file for Power BI

In [12]:
df.to_csv("iqvia_market_access.csv", index=False)
print("File saved successfully!")

File saved successfully!
